# Retrain Stage 1 -- with the bug fixes (meeting prep)

Runs the `meeting-prep` branch: Task 2's time-matched features plus the fixes below. Since these are bug fixes
(the old model was trained on partly wrong inputs), the numbers here are a **new baseline**, not something to
compare ADE/FDE against earlier checkpoints.

| | before | now |
|---|---|---|
| agents fed to the model | valid in *every* history frame, any distance, first 32 in array order | valid **now**, within **50 m**, **nearest 32**; dropouts in the 1 s history interpolated + flagged (7th feature) |
| map points | first **200** within 80 m in *array* order (arbitrary subset) | **all** within 80 m, nearest-first (cap 6000, measured below) |
| objects loaded | `max_num_objects=32` -- first 32 in file order, SDC not guaranteed | `128` (all of WOMD) + SDC presence checked |
| ego's own history | invalid frames used as-is (sentinel values) | gap-filled like agents |
| LR schedule | `T_max=5` but trained 10 epochs -> LR rose again for epochs 6-9 | `T_max = NUM_EPOCHS` |
| test metrics | computed on the *last* epoch's model | computed on the **saved best** checkpoint |

**Runtime**: roughly 1-1.5 h on a T4 for 10 epochs x 2000 scenarios (the map branch now sees far more
points). After epoch 0 the training cell prints a projection -- raise `TRAIN_SCENARIOS` next time if you have
the budget: each epoch re-reads the same first N training scenarios (the loader doesn't shuffle), so more
scenarios = more distinct data.

Outputs: `weights/stage1_weights_roi.pth`, `results/retrain_meeting_prep.json`, and a markdown table at the end
for the README.

## Setup

In [ ]:
from google.colab import auth
auth.authenticate_user()
!git config --global user.email "mayankarya510@gmail.com"
!git config --global user.name "meyenk"

In [ ]:
!pip install -q git+https://github.com/waymo-research/waymax.git@main#egg=waymo-waymax

Clones the `meeting-prep` branch, not `main` -- the changes this notebook tests aren't merged yet.

In [ ]:
!git clone -b meeting-prep https://github.com/meyenk/waymax_experiment.git
%cd waymax_experiment
!git log --oneline -3

**(Optional) push access** -- only needed to push results/weights back from here. Skip if you'd rather download and commit yourself.

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git remote set-url origin https://{token}@github.com/meyenk/waymax_experiment.git

## Sanity check (synthetic tests, no Waymax needed)
If any of these fail, stop -- the checkout is wrong or something regressed.

In [ ]:
!python -m tests.test_features
!python -m tests.test_transforms
!python -m tests.test_mpc
!python -m tests.test_closed_loop

## Data + diagnostics (before spending GPU time)

In [ ]:
import dataclasses, json, os, time
import numpy as np
import torch
import jax
from waymax import dataloader, config as _config

from src import features as F
from src.dataset import val_test_split
jax.config.update("jax_enable_x64", True)

# max_num_objects=128 = every object WOMD has. The old notebooks used 32, and
# Waymax keeps the first 32 in FILE order (not nearest, and with no guarantee
# the SDC is among them -- see the check below).
MAX_NUM_OBJECTS = 128
train_config = dataclasses.replace(
    _config.WOD_1_1_0_TRAINING,
    path="gs://waymo_open_dataset_motion_v_1_1_0/uncompressed/tf_example/training/training_tfexample.tfrecord@1000",
    max_num_objects=MAX_NUM_OBJECTS,
)
val_config = dataclasses.replace(
    _config.WOD_1_1_0_VALIDATION,
    path="gs://waymo_open_dataset_motion_v_1_1_0/uncompressed/tf_example/validation/validation_tfexample.tfrecord@150",
    max_num_objects=MAX_NUM_OBJECTS,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {device}")
print(f"ROI: agents within {F.AGENT_RADIUS} m (max {F.MAX_AGENTS}), "
      f"map within {F.MAP_RADIUS} m (max {F.MAX_MAP_POINTS} points), lights max {F.MAX_LIGHTS}")

print("loading validation scenarios (splitting into val/test, 70/30)...")
val_scenarios, test_scenarios = [], []
for i, scenario in enumerate(dataloader.simulator_state_generator(config=val_config)):
    if len(val_scenarios) + len(test_scenarios) >= 1000:
        break
    (val_scenarios if val_test_split(i) == "val" else test_scenarios).append(scenario)
print(f"  val scenarios: {len(val_scenarios)}, test scenarios: {len(test_scenarios)}")

### Was the SDC ever missing under the old `max_num_objects=32`?
Reloads 300 validation scenarios with the old cap and counts how many no longer contain the SDC. For those,
`np.argmax(is_sdc)` returned 0 -- the old pipeline silently treated some other object as the ego.

In [ ]:
old_cfg = dataclasses.replace(val_config, max_num_objects=32)
n_checked, n_missing = 0, 0
for i, s in enumerate(dataloader.simulator_state_generator(config=old_cfg)):
    if i >= 300:
        break
    n_checked += 1
    n_missing += int(not np.array(s.object_metadata.is_sdc).any())
n_missing_128 = sum(int(not np.array(s.object_metadata.is_sdc).any()) for s in val_scenarios + test_scenarios)
print(f"max_num_objects=32 : SDC missing in {n_missing}/{n_checked} scenarios ({100*n_missing/max(n_checked,1):.1f}%)")
print(f"max_num_objects=128: SDC missing in {n_missing_128}/{len(val_scenarios)+len(test_scenarios)}")
diagnostics = {"sdc_missing_cap32": [n_missing, n_checked], "sdc_missing_cap128": n_missing_128}

### What does the new selection actually change?
Over 150 validation scenarios, per decision instant: agents picked under the new vs old rule, how many of the
new picks the old rule would have **dropped** (a dropout in the window), how many agents the old rule fed the
model from **outside** the ROI, how much of the history is filled, and how many map points fall in range.

**Check `map cap binds`**: if it's more than a few %, raise `F.MAX_MAP_POINTS` here (it isn't part of the
weights, so any value is fine) -- the aim is to cover everything in the radius.

In [ ]:
from src.visualize import selection_report
rep = selection_report(val_scenarios[:150])
pct = lambda a: np.percentile(a, [50, 90, 99]).round(0).astype(int).tolist()
summary = {
    "agents picked (new), mean": float(rep["agents_new"].mean()),
    "agents fed (old rule), mean": float(rep["agents_old"].mean()),
    "new picks the old rule dropped (dropout), mean": float(rep["agents_rescued"].mean()),
    "decision instants with >=1 rescued agent (%)": float(100 * (rep["agents_rescued"] > 0).mean()),
    "old-rule agents outside the ROI, mean": float(rep["agents_far_old"].mean()),
    "history frames filled among picked agents (%)": float(100 * rep["frac_frames_filled"].mean()),
    "map points in radius p50/p90/p99": pct(rep["map_in_radius"]),
    "map cap binds (% of instants)": float(100 * (rep["map_in_radius"] > F.MAX_MAP_POINTS).mean()),
}
for k, v in summary.items():
    print(f"  {k:52s} {v if isinstance(v, list) else round(v, 2)}")
diagnostics["selection"] = summary

### See it
Left: world frame -- coloured map points/agents are exactly what the model gets; red rings = agents the old
rule would have dropped; `x` = filled frames. Right: the agent tensor itself, in the ego frame.

In [ ]:
from src.visualize import plot_selection, pick_dynamic_scenario, pick_turning_scenario
demo = pick_dynamic_scenario(val_scenarios, min_displacement=35.0)
turn = pick_turning_scenario(val_scenarios, min_heading_change_deg=25.0, min_displacement=5.0)
for s in (demo, turn):
    for t in (19, 49):
        plot_selection(s, t)

## Train

In [ ]:
from src.model import Stage1Model
from src.train import run_epoch
import src.train as train_mod

NUM_EPOCHS = 10
TRAIN_SCENARIOS = 2000   # same as the earlier runs; raise if you have the time budget
BATCH_SIZE = 32
LR = 1e-3
SEED = 0
os.makedirs("weights", exist_ok=True)
os.makedirs("results", exist_ok=True)
CKPT = "weights/stage1_weights_roi.pth"

torch.manual_seed(SEED)
np.random.seed(SEED)
model = Stage1Model(hidden_dim=64, future_len=30).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)  # matches the loop now

history = {"train": [], "val": [], "lr": []}
best = {"val_loss": float("inf"), "epoch": None}
t_start = time.time()
for epoch in range(NUM_EPOCHS):
    print(f"\n--- epoch {epoch} ---")
    history["lr"].append(optimizer.param_groups[0]["lr"])
    tr = run_epoch(model, optimizer, dataloader.simulator_state_generator(config=train_config),
                   max_scenarios=TRAIN_SCENARIOS, batch_size=BATCH_SIZE, train=True,
                   device=device, verbose_every=25)
    va = run_epoch(model, optimizer, val_scenarios, max_scenarios=len(val_scenarios),
                   batch_size=BATCH_SIZE, train=False, device=device, verbose_every=0)
    scheduler.step()
    history["train"].append(tr); history["val"].append(va)
    flag = ""
    if va["loss"] < best["val_loss"]:
        best = {"val_loss": va["loss"], "epoch": epoch}
        torch.save(model.state_dict(), CKPT)
        flag = "  -> new best, saved"
    print(f"train loss={tr['loss']:.4f} ade={tr['ade']:.3f} fde={tr['fde']:.3f} | "
          f"val loss={va['loss']:.4f} ade={va['ade']:.3f} fde={va['fde']:.3f} | lr={history['lr'][-1]:.2e}{flag}")
    if epoch == 0:
        per_epoch = time.time() - t_start
        print(f"  (epoch took {per_epoch / 60:.1f} min -> ~{per_epoch * NUM_EPOCHS / 60:.0f} min total)")
print(f"\nmap cap bound on {100 * tr['map_truncated_frac']:.1f}% of training examples (last epoch)")
print(f"scenarios skipped for a missing SDC: {train_mod.SKIPPED_NO_SDC}")
print(f"best epoch: {best['epoch']} (val loss {best['val_loss']:.4f}) -- saved to {CKPT}")

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for split, ls in (("train", "-"), ("val", "--")):
    axes[0].plot([h["loss"] for h in history[split]], ls=ls, label=split)
    axes[1].plot([h["fde"] for h in history[split]], ls=ls, label=split)
axes[2].plot(history["lr"])
for ax, t in zip(axes, ("MSE loss", "FDE (m)", "learning rate (falls monotonically now)")):
    ax.set_title(t); ax.set_xlabel("epoch"); ax.legend(fontsize=8) if ax is not axes[2] else None
plt.tight_layout(); plt.show()

## Test set -- on the saved best checkpoint
Overall, and split into turning vs straight scenarios (turns are where the model has struggled).

In [ ]:
from src.visualize import is_turning_scenario
model.load_state_dict(torch.load(CKPT, map_location=device))
model.eval()
turning = [s for s in test_scenarios if is_turning_scenario(s)]
straight = [s for s in test_scenarios if not is_turning_scenario(s)]
print(f"test: {len(test_scenarios)} scenarios ({len(turning)} turning, {len(straight)} straight)")

test_results = {}
for subset_name, subset in (("all", test_scenarios), ("turning", turning), ("straight", straight)):
    r = run_epoch(model, None, subset, max_scenarios=len(subset), batch_size=BATCH_SIZE,
                  train=False, device=device, verbose_every=0)
    test_results[subset_name] = r
    print(f"  {subset_name:8s} loss={r['loss']:.4f} ADE={r['ade']:.3f} m FDE={r['fde']:.3f} m  (n={r['n_examples']})")

## Does the model actually use the scene? (branch ablation)
Each row hides whole encoder branches at inference (their items are masked out, so the pool returns zeros).
No retraining -- same checkpoint. If hiding agents+map+lights barely moves ADE/FDE, the model is mostly
extrapolating its own speed, which would explain the "keeps doing what it's doing" bias.

In [ ]:
ablations = [(), ("agents",), ("map",), ("lights",), ("agents", "map", "lights")]
ablation_results = {}
for subset_name, subset in (("all", test_scenarios), ("turning", turning)):
    for ab in ablations:
        r = run_epoch(model, None, subset, max_scenarios=len(subset), batch_size=BATCH_SIZE,
                      train=False, device=device, verbose_every=0, ablate=ab)
        key = "+".join(ab) or "none"
        ablation_results[f"{subset_name}/{key}"] = r
        print(f"  {subset_name:8s} hide {key:22s} ADE={r['ade']:.3f} m  FDE={r['fde']:.3f} m")

## Look at predictions + what the trained model attends to

In [ ]:
from src.visualize import plot_world_frame_snapshots, play_scenario_clip
turn_test = pick_turning_scenario(test_scenarios, min_heading_change_deg=25.0, min_displacement=5.0)
play_scenario_clip(turn_test)
plot_world_frame_snapshots(model, turn_test, num_snapshots=6, window=80.0, device=device)
# colours = the model's learned relevance weight per agent / map point
plot_selection(turn_test, 39, model=model, device=device)

## Save results + README table

In [ ]:
results = {
    "config": {"num_epochs": NUM_EPOCHS, "train_scenarios": TRAIN_SCENARIOS, "batch_size": BATCH_SIZE,
               "lr": LR, "seed": SEED, "max_num_objects": MAX_NUM_OBJECTS,
               "agent_radius": F.AGENT_RADIUS, "max_agents": F.MAX_AGENTS, "map_radius": F.MAP_RADIUS,
               "max_map_points": F.MAX_MAP_POINTS},
    "best": best, "history": history, "test": test_results, "ablation": ablation_results,
    "diagnostics": diagnostics,
}
with open("results/retrain_meeting_prep.json", "w") as f:
    json.dump(results, f, indent=1, default=float)

lines = [f"Trained {NUM_EPOCHS} epochs on {TRAIN_SCENARIOS} scenarios, best epoch {best['epoch']}.", "",
         "| test subset | ADE (m) | FDE (m) | examples |", "|---|---|---|---|"]
for sub, r in test_results.items():
    lines.append(f"| {sub} | {r['ade']:.3f} | {r['fde']:.3f} | {r['n_examples']} |")
lines += ["", "| hidden branches (all test) | ADE (m) | FDE (m) |", "|---|---|---|"]
for k, r in ablation_results.items():
    if k.startswith("all/"):
        lines.append(f"| {k[4:]} | {r['ade']:.3f} | {r['fde']:.3f} |")
print("\n".join(lines))

**(Optional) commit + push the checkpoint and results** -- requires the push-access cell above.

In [ ]:
!git add weights/stage1_weights_roi.pth results/retrain_meeting_prep.json
!git commit -m "Retrain with ROI selection + bug fixes"
!git push origin meeting-prep

If you didn't push: download `weights/stage1_weights_roi.pth` (Files panel) -- the closed-loop notebook needs it.